In [8]:
import pandas as pd

from mlxtend.frequent_patterns import apriori, association_rules

# -----------------------
# Step 1: Create dataset
# -----------------------
dataset = [
    ['milk', 'bread', 'nuts', 'apple'],
    ['milk', 'bread', 'cornflakes'],
    ['milk', 'bread', 'nuts'],
    ['bread', 'cornflakes', 'apple'],
    ['milk', 'bread', 'cornflakes', 'apple']
]

# Convert dataset into a one-hot encoded DataFrame
from mlxtend.preprocessing import TransactionEncoder
te = TransactionEncoder()
te_array = te.fit(dataset).transform(dataset)
df = pd.DataFrame(te_array, columns=te.columns_)

print("Transaction Data (One-Hot Encoded):")
print(df)

# -----------------------
# Step 2: Apply Apriori
# -----------------------
frequent_itemsets = apriori(df, min_support=0.4, use_colnames=True)
print("\nFrequent Itemsets using Apriori:")
print(frequent_itemsets)

# Generate association rules
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.0)
print("\nAssociation Rules:")
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']])

# -----------------------
# Step 3: Naive Algorithm
# (Brute-force Frequent Itemsets)
# -----------------------
from itertools import combinations

def naive_frequent_itemsets(df, min_support=0.4):
    items = df.columns
    n_transactions = len(df)
    results = []

    for size in range(1, len(items)+1):
        for combo in combinations(items, size):
            support = (df[list(combo)].all(axis=1).sum()) / n_transactions
            if support >= min_support:
                results.append({"itemsets": combo, "support": support})
    return pd.DataFrame(results)

naive_itemsets = naive_frequent_itemsets(df, min_support=0.4)
print("\nFrequent Itemsets using Naive Algorithm:")
print(naive_itemsets)

Transaction Data (One-Hot Encoded):
   apple  bread  cornflakes   milk   nuts
0   True   True       False   True   True
1  False   True        True   True  False
2  False   True       False   True   True
3   True   True        True  False  False
4   True   True        True   True  False

Frequent Itemsets using Apriori:
    support                    itemsets
0       0.6                     (apple)
1       1.0                     (bread)
2       0.6                (cornflakes)
3       0.8                      (milk)
4       0.4                      (nuts)
5       0.6              (bread, apple)
6       0.4         (apple, cornflakes)
7       0.4               (milk, apple)
8       0.6         (bread, cornflakes)
9       0.8               (milk, bread)
10      0.4               (nuts, bread)
11      0.4          (milk, cornflakes)
12      0.4                (milk, nuts)
13      0.4  (bread, apple, cornflakes)
14      0.4        (milk, bread, apple)
15      0.4   (milk, bread, cornflakes

/home/cse/anaconda3/lib/python3.12/site-packages/mlxtend/frequent_patterns/association_rules.py:186: RuntimeWarning: invalid value encountered in divide
  cert_metric = np.where(certainty_denom == 0, 0, certainty_num / certainty_denom)


In [9]:
import pandas as pd

# Dataset
data = {
    "Antecedent": ["A","A","A","A","B","B","B"],
    "Consequent": [0,0,1,0,1,0,1]
}
df = pd.DataFrame(data)

total = len(df)

# A → 0
lhs_count = len(df[df["Antecedent"]=="A"])
both_count = len(df[(df["Antecedent"]=="A") & (df["Consequent"]==0)])
rhs_count = len(df[df["Consequent"]==0])
support_A0 = both_count / total
confidence_A0 = both_count / lhs_count
lift_A0 = confidence_A0 / (rhs_count / total)

# A → 1
lhs_count = len(df[df["Antecedent"]=="A"])
both_count = len(df[(df["Antecedent"]=="A") & (df["Consequent"]==1)])
rhs_count = len(df[df["Consequent"]==1])
support_A1 = both_count / total
confidence_A1 = both_count / lhs_count
lift_A1 = confidence_A1 / (rhs_count / total)

# B → 0
lhs_count = len(df[df["Antecedent"]=="B"])
both_count = len(df[(df["Antecedent"]=="B") & (df["Consequent"]==0)])
rhs_count = len(df[df["Consequent"]==0])
support_B0 = both_count / total
confidence_B0 = both_count / lhs_count
lift_B0 = confidence_B0 / (rhs_count / total)

# B → 1
lhs_count = len(df[df["Antecedent"]=="B"])
both_count = len(df[(df["Antecedent"]=="B") & (df["Consequent"]==1)])
rhs_count = len(df[df["Consequent"]==1])
support_B1 = both_count / total
confidence_B1 = both_count / lhs_count
lift_B1 = confidence_B1 / (rhs_count / total)

# Display results
print("A → 0:", support_A0, confidence_A0, lift_A0)
print("A → 1:", support_A1, confidence_A1, lift_A1)
print("B → 0:", support_B0, confidence_B0, lift_B0)
print("B → 1:", support_B1, confidence_B1, lift_B1)



A → 0: 0.42857142857142855 0.75 1.3125
A → 1: 0.14285714285714285 0.25 0.5833333333333334
B → 0: 0.14285714285714285 0.3333333333333333 0.5833333333333334
B → 1: 0.2857142857142857 0.6666666666666666 1.5555555555555556


In [1]:
from itertools import combinations
from collections import defaultdict
transactions = [
    ["milk", "bread", "butter"],       
    ["bread", "butter"],               
    ["milk", "bread"],             
    ["milk", "bread", "butter", "eggs"],
    ["bread", "butter", "eggs"]        
]

total = len(transactions)
item_counts = defaultdict(int)
for t in transactions:
    for r in range(1, len(t)+1):
        for combo in combinations(sorted(t), r):
            item_counts[combo] += 1
def calc_rule(lhs, rhs):
    lhs_tuple = tuple(sorted(lhs))
    both_tuple = tuple(sorted(lhs + rhs))
    rhs_tuple = tuple(sorted(rhs))

    support = item_counts[both_tuple] / total
    confidence = item_counts[both_tuple] / item_counts[lhs_tuple]
    lift = confidence / (item_counts[rhs_tuple] / total)
    return support, confidence, lift
rules = [
    (["milk"], ["butter"]),
    (["milk"], ["bread"]),
    (["milk","bread"], ["butter"])
]
for lhs, rhs in rules:
    support, confidence, lift = calc_rule(lhs, rhs)
    print(f"{lhs} → {rhs} | Support: {support:.3f}, Confidence: {confidence:.3f}, Lift: {lift:.3f}")

['milk'] → ['butter'] | Support: 0.400, Confidence: 0.667, Lift: 0.833
['milk'] → ['bread'] | Support: 0.600, Confidence: 1.000, Lift: 1.000
['milk', 'bread'] → ['butter'] | Support: 0.400, Confidence: 0.667, Lift: 0.833
